In [1]:
from openff.toolkit import Molecule, ForceField
from openff.interchange import Interchange
import openmm.app
import openmm
import openmm.unit 


In [10]:
inpcrd1 = openmm.app.AmberInpcrdFile('monomer.inpcrd')
prmtop1 = openmm.app.AmberPrmtopFile('monomer.prmtop', 
                                     periodicBoxVectors=inpcrd1.boxVectors)
system1 = prmtop1.createSystem(nonbondedMethod=openmm.app.PME, 
                               nonbondedCutoff=9 * openmm.unit.angstrom, 
                               switchDistance=8 * openmm.unit.angstrom,
                               constraints=openmm.app.HBonds)
i1 = Interchange.from_openmm(system1, 
                             prmtop1.topology, 
                             positions=inpcrd1.positions, 
                             box_vectors=inpcrd1.boxVectors)

In [11]:
inpcrd2 = openmm.app.AmberInpcrdFile('ALA_ALA.inpcrd')
prmtop2 = openmm.app.AmberPrmtopFile('ALA_ALA.prmtop', 
                                     periodicBoxVectors=inpcrd2.boxVectors)
system2 = prmtop2.createSystem(nonbondedMethod=openmm.app.PME, 
                               nonbondedCutoff=9 * openmm.unit.angstrom, 
                               switchDistance=8 * openmm.unit.angstrom,
                               constraints=openmm.app.HBonds)
i2 = Interchange.from_openmm(system2, 
                             prmtop2.topology, 
                             positions=inpcrd2.positions, 
                             box_vectors=inpcrd2.boxVectors)

In [12]:
offmol = Molecule.from_smiles("CCO")
offmol.generate_conformers()
ff = ForceField('openff-2.3.0-rc2.offxml')
i3 = ff.create_interchange(offmol.to_topology())

In [18]:
comb_interchange = i1.combine(i2.combine(i3))

In [19]:
integrator = openmm.LangevinIntegrator(
    300 * openmm.unit.kelvin,
    1 / openmm.unit.picosecond,
    0.002 * openmm.unit.picoseconds,
)

simulation = comb_interchange.to_openmm_simulation(integrator=integrator)

KeyError: SingleAtomChargeTopologyKey(this_atom_index=50, extras={})

In [20]:
comb_interchange = i3.combine(i2.combine(i1))

UnsupportedCombinationError: Combination failed due to key collision on id='Constraint0_DUPLICATE' mult=None associated_handler=None bond_order=None virtual_site_type=None cosmetic_attributes={}. Some keys already have _DUPLICATE appended to their ID. Please report this issue if you need this functionality.